# 📌 Customer Segmentation Project

**Problem Statement:** Customer segmentation

**Goal:** Group customers based on behavior using RFM analysis and K-Means clustering.

**Tools:** Python, pandas, scikit-learn, matplotlib, seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 📂 Load Dataset

In [ ]:
from pathlib import Path

data_path = Path("data/customers.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

df = pd.read_csv(data_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

## 🔍 Data Exploration (EDA)

In [ ]:
print(df.info())
display(df.describe(include="all"))
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["Revenue"] if "Revenue" in df.columns else df.select_dtypes(include="number").iloc[:,0], kde=True)
plt.title("Revenue Distribution")
plt.show()

In [ ]:
if "CustomerID" in df.columns:
    df.groupby("CustomerID").size().hist()
    plt.title("Orders per Customer")
    plt.show()

In [ ]:
if "Category" in df.columns:
    sns.countplot(y=df["Category"])
    plt.title("Category Distribution")
    plt.show()

## 🧹 Data Preprocessing

In [ ]:
required_cols = ["OrderDate", "Quantity", "UnitPrice", "CustomerID"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df = df.dropna(subset=["OrderDate", "CustomerID", "TotalAmount"])
print("Rows after cleaning:", len(df))

## 📊 RFM Feature Engineering

In [ ]:
snapshot_date = df["OrderDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg({
    "OrderDate": lambda x: (snapshot_date - x.max()).days,
    "OrderID": "count",
    "TotalAmount": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]
rfm.reset_index(inplace=True)
rfm.head()

## 📉 Data Scaling

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recency","Frequency","Monetary"]])

## 📊 Elbow Method

In [ ]:
inertia = []
silhouette_scores = []
K = range(2, 10)

for k in K:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(rfm_scaled)
    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))

best_k = list(K)[silhouette_scores.index(max(silhouette_scores))]
print("Best k (silhouette):", best_k)

plt.plot(list(K), inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("Inertia")
plt.show()

## 🤖 Apply KMeans Clustering

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
rfm.head()

## 📈 Cluster Analysis

In [ ]:
cluster_summary = rfm.groupby("Cluster").mean(numeric_only=True)
cluster_summary

Suggested labels:

- Cluster 0 → High Value Customers
- Cluster 1 → Loyal Customers
- Cluster 2 → At-Risk Customers

## 📊 Visualization of Segments

In [ ]:
sns.scatterplot(x=rfm["Recency"], y=rfm["Monetary"], hue=rfm["Cluster"], palette="Set2")
plt.title("Customer Segments")
plt.show()

## 📁 Save Final Output

In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

out_file = output_dir / "clusters.csv"
rfm.to_csv(out_file, index=False)
print(f"Saved: {out_file.resolve()}")

## 🧠 Insights / Conclusion

- Identify the segment generating the highest revenue.
- Identify inactive or lost customers.
- Create retention and win-back campaigns.

**Examples:**
- High spenders → Target with premium offers.
- Loyal customers → Reward with loyalty programs.
- At-risk customers → Launch win-back campaigns.